### imports

In [1]:
import json
import pickle
from datetime import datetime as dt

import numpy as np
import pandas as pd

### functions

In [2]:
# use POSIX timestamp of first experiment page as start time --
# `beginhit` field records when prep screen was loaded, which was
# often done in advance
exp_start_time = lambda x: x['data'][0]['dateTime']

# workaround for the datetime conversion from the google forms -- 
# Google forms convert all dates to current time zone upon 
# downloading, so participants collected during EDT are shown 
# as EST equivalent. Pandas' default date_parser doesn't handle 
# this weird behavior correctly
parse_posix_ms = lambda x: pd.to_datetime(x).tz_localize(None).tz_localize('EST').timestamp()*1000

# a bit of a hack to allow merging (on columns) two dataframes 
# when the key column has duplicates
def merge_with_duplicates(df, other, on=None, **kwargs):
    df['_'] = df.groupby(on).cumcount()
    other['_'] = other.groupby(on).cumcount()
    if isinstance(on, str):
        on = [on] + ['_']
    else:
        on = list(on) + ['_']
    return df.merge(other, on=on, **kwargs).drop(columns='_')

### load data

In [3]:
# load in psiturk data
rm1df = pd.read_json('../../../data/db/exported/room1-3.8.19.json')
rm2df = pd.read_json('../../../data/db/exported/room2-3.5.19.json')

# drop various debugging runs
rm1df = rm1df[rm1df.status != 1].reset_index(drop=True)
rm2df = rm2df[rm2df.status != 1].reset_index(drop=True)

# remove test runs for each room
rm1df = rm1df.loc[1:]
rm2df = rm2df.loc[1:]

# keep relevant columns
rm1df = rm1df[['uniqueid','datastring']]
rm2df = rm2df[['uniqueid','datastring']]

# load json string
rm1df['datastring'] = rm1df['datastring'].apply(json.loads)
rm2df['datastring'] = rm2df['datastring'].apply(json.loads)

# record start time
rm1df['beginhit'] = rm1df['datastring'].apply(exp_start_time)
rm2df['beginhit'] = rm2df['datastring'].apply(exp_start_time)

# make sure using screen 1 start time didn't change order
assert np.array_equal(rm1df.index, rm1df.sort_values(['beginhit']).index)
assert np.array_equal(rm2df.index, rm2df.sort_values(['beginhit']).index)

# record test room
rm1df['testroom'] = 1
rm2df['testroom'] = 2


# concatenate testroom dataframes, order by start time
expdf = pd.concat([rm1df, rm2df], ignore_index=True).sort_values('beginhit').reset_index(drop=True)

In [4]:
expdf.head()

,uniqueid,datastring,beginhit,testroom
0,debugIEH2T:debugDLVLJ,"{'condition': 0, 'counterbalance': 0, 'assignm...",1539368162836,1
1,debugBUnNA:debugLtZcs,"{'condition': 0, 'counterbalance': 0, 'assignm...",1539371956776,1
2,debugYQfMB:debugxg7il,"{'condition': 0, 'counterbalance': 0, 'assignm...",1539372566510,2
3,debugd1YD1:debug4FrAg,"{'condition': 0, 'counterbalance': 0, 'assignm...",1539375821845,1
4,debug92cgv:debugvdAIT,"{'condition': 0, 'counterbalance': 0, 'assignm...",1539376317256,2


## load pre- & post-questionnaire responses

In [5]:
# load, convert timestamps, exclude test runs
preqdf = pd.read_csv('../../../data/google-form-data/Pre-experiment Questionnaire.csv')
preqdf = preqdf.rename(index=str, columns={'Timestamp':'preqtime'})
preqdf['preqtime'] = preqdf['preqtime'].apply(parse_posix_ms)
preqdf = preqdf.dropna(subset=['Subject ID']).reset_index(drop=True)

postqdf = pd.read_csv('../../../data/google-form-data/Post-experiment questionnaire.csv')
postqdf = postqdf.rename(index=str, columns={'Timestamp':'postqtime'})
postqdf['postqtime'] = postqdf['postqtime'].apply(parse_posix_ms)
postqdf = postqdf.dropna(subset=['Subject ID']).reset_index(drop=True)


# exclude entries where experiment was stopped partway 
# through -- causes there to be an entry in pre-questionnaire 
# but not experiment database or post-questionnaire
errs_ses1 = []
errs_ses2 = []
errs_ses1.append('MD-1011318-A-05')    # computer backing up during experiment caused crash
errs_ses1.append('MD-020119-A-01')     # participant accidentally hit button to stop recall
errs_ses1.append('MD-020119-B-01')     # participant chose to drop out due to illness
errs_ses2.append('MD-101218-B-04')     # Notification caused experiment to exit full-screen

preqdf = preqdf[~preqdf['Subject ID'].isin(errs_ses1)]

for s2_err in errs_ses2:
    row = preqdf.loc[preqdf['Subject ID'] == s2_err].index[-1]
    preqdf.drop(row, inplace=True)

preqdf.reset_index(drop=True, inplace=True)

### some quick checks

In [6]:
# check conversion to posix time for bugs that could've disrupted order
assert np.array_equal(preqdf.index, preqdf.sort_values(['preqtime']).index)
assert np.array_equal(postqdf.index, postqdf.sort_values(['postqtime']).index)

# check equal number of entries in questionnaires and experiment db
assert expdf.shape[0] == preqdf.shape[0] == postqdf.shape[0]

In [7]:
# # spot-check any pairings with time differences > 30s
# for ix in range(expdf.shape[0]):
#     diff = (expdf.loc[ix, 'beginhit'] - preqdf.loc[ix, 'preqtime']) / 1000
#     print(diff)
#     if diff < 0 or diff > 30 or preqdf.loc[ix, 'Subject ID'] == 'MD-102118-A-01':
#         print('\n\n')
#         print(preqdf.loc[ix, 'Subject ID'], f' .... row number: {ix}')
#         print('\tpreq time vs previous expdf time:')
#         print(f"\t\t{(expdf.loc[ix-1, 'beginhit'] - preqdf.loc[ix, 'preqtime']) / 1000}")
#         print('\tpostq time vs next expdf time:')
#         print(f"\t\t{(expdf.loc[ix+1, 'beginhit'] - preqdf.loc[ix, 'preqtime']) / 1000}")
#         print(f"psiturk ID : {expdf.loc[ix, 'uniqueid']}")
#         print('\n\n')

## combine experiment data and questionnaire data

In [8]:
expdf = pd.concat([expdf, preqdf], axis=1)
expdf = merge_with_duplicates(expdf, postqdf, on='Subject ID')

In [9]:
expdf.head()

,uniqueid,datastring,beginhit,testroom,preqtime,Subject ID,"Outside of this study, have you ever watched an episode of either of the TV shows ""Atlanta"" or ""Arrested Development?""",Is English your first language?,Do you have any hearing or speech impairments?,Do you have normal color vision?,...,What is/was your major?,How many hours of sleep did you get last night?,How many cups of coffee have you had today?,How alert are you feeling?,postqtime,How engaging did you find the episode?,How easy/difficult was it to follow the episode?,How well do you feel you recalled the events of the episode?,How well do you feel you learned the characters' names over the course of the episode?,How tired do you feel?
0,debugIEH2T:debugDLVLJ,"{'condition': 0, 'counterbalance': 0, 'assignm...",1539368162836,1,1.539368e+12,MD-101218-A-01,I've never watched either one,Yes,No,Yes,...,undeclared,7.0,1.0,A little alert,1.539372e+12,Very engaging,Somewhat easy,Very well,Very well,A little tired
1,debugBUnNA:debugLtZcs,"{'condition': 0, 'counterbalance': 0, 'assignm...",1539371956776,1,1.539372e+12,MD-101218-B-01,I've never watched either one,Yes,No,Yes,...,undeclared,5.0,0.0,A little sluggish,1.539375e+12,A little engaging,Somewhat easy,Very well,Somewhat well,A little tired
2,debugYQfMB:debugxg7il,"{'condition': 0, 'counterbalance': 0, 'assignm...",1539372566510,2,1.539373e+12,MD-101218-A-02,I've never watched either one,Yes,No,Yes,...,undeclared,7.0,2.0,A little alert,1.539376e+12,Very engaging,Somewhat easy,Somewhat well,Somewhat well,A little tired
3,debugd1YD1:debug4FrAg,"{'condition': 0, 'counterbalance': 0, 'assignm...",1539375821845,1,1.539376e+12,MD-101218-B-02,I've never watched either one,Yes,No,Yes,...,neuroscience,9.0,0.0,A little alert,1.539379e+12,Very engaging,Very easy,Very well,Somewhat well,A little alert
4,debug92cgv:debugvdAIT,"{'condition': 0, 'counterbalance': 0, 'assignm...",1539376317256,2,1.539376e+12,MD-101218-A-03,I've never watched either one,Yes,No,Yes,...,Sociology,7.0,0.0,A little alert,1.539379e+12,Very engaging,Somewhat easy,Very well,Somewhat well,Very alert


### do some quick corrections and drop excluded participants

In [10]:
# correct some typos in the google form...
typos = {
    'MD--22819-B-01' : 'MD-022819-B-01',
    'MD-101218-A-06' : 'MD-102218-A-06',
    'MD-102118-B-06' : 'MD-102218-B-06',
    'MD-011319-A-02' : 'MD-013119-A-02'
}

expdf.replace(typos, inplace=True)

In [11]:
# ... and fix a repeated ID manually
expdf.loc[expdf.loc[expdf['Subject ID'] == 'MD-101218-A-03'].index[-1], 'Subject ID'] = 'MD-102018-A-03'

In [12]:
# exclude participants who did not complete the task or did not return for session 2
dropids = ['MD-020119-B-03', 'MD-102218-B-06', 'MD-101318-A-01', 'MD-013119-A-01', 
           'MD-102218-B-04', 'MD-102318-A-01', 'MD-022019-B-01']

# other exclusions
dropids.append('MD-101218-B-04')    # Docker crashed during session 2
dropids.append('MD-102218-A-05')    # audio recording paused during task
dropids.append('MD-102218-A-04')    # self-reported difficulty with task due to migraine
dropids.append('MD-020719-B-01')    # reported issues with computer audio during task

expdf = expdf[~expdf['Subject ID'].isin(dropids)].reset_index(drop=True)

In [13]:
# make sure all Subject IDs occur exactly twice
assert (expdf['Subject ID'].value_counts() == 2).all()

## format & save across-session ID mappings for analyses

In [14]:
id_maps = {sid : list(expdf.loc[expdf['Subject ID'] == sid, 'uniqueid'].values) 
           for sid in expdf['Subject ID'].unique()}
id_maps = pd.DataFrame.from_dict(id_maps, orient='index', columns=['session 1', 'session 2'])

In [15]:
with open('../../../data/pickles/expdf.p', 'wb') as file:
    pickle.dump(expdf, file)

with open('../../../data/pickles/id_maps.p', 'wb') as file:
    pickle.dump(id_maps, file)